# 阶段 00 · Colab 一键环境准备

每个新 Colab 会话**先跑这个 notebook**, 解决"重下模型 + 重装依赖"的麻烦。

## 做了三件事
1. 挂载 Drive, 把 HuggingFace 模型缓存指到 Drive → **模型只下一次, 以后复用**;
2. 克隆/更新项目代码;
3. 安装 unsloth 及依赖 (装完需重启会话)。

## 使用顺序
- 运行 [A1]~[A3] → **重启会话** → 运行 [B1]~[B2] 验证。
- 之后即可跑各阶段的 notebook (05 训练 / 06 评测 / ...)。

> 提示: 运行时记得先选 T4 GPU (代码执行程序 → 更改运行时类型)。

In [ ]:
# [A1] 挂载 Drive + 把模型缓存指到 Drive (必须在 import 任何 HF 库之前设置)
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"   # 模型/数据集缓存 -> Drive
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
print("HF_HOME =", os.environ["HF_HOME"])

In [ ]:
# [A2] 克隆或更新项目代码
import os
if os.path.exists("/content/Lora"):
    %cd /content/Lora
    !git pull
else:
    %cd /content
    !git clone https://github.com/xinsu0828-max/Lora.git
    %cd /content/Lora
!ls

In [ ]:
# [A3] 安装依赖 (经验证的配方); 装完必须重启会话!
!pip install unsloth
!pip install --upgrade --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo
!pip install -q fastapi uvicorn openai

print("\n>>> 安装完成。请现在: 菜单 → 代码执行程序 → 重启会话, 然后运行 [B1] <<<")

---
### ⏸ 在这里重启会话 (代码执行程序 → 重启会话), 然后运行下面的 [B1]、[B2]
---

In [ ]:
# [B1] 重启后: 重新挂载 + 重设缓存 + 进项目 + 验证 unsloth
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"   # 每次会话都要在 import 前重设
%cd /content/Lora

import unsloth
print("unsloth ok:", unsloth.__version__, "| HF_HOME:", os.environ["HF_HOME"])

In [ ]:
# [B2] (可选) 从 Drive 恢复之前训练好的权重, 跳过重训
import shutil, os
restore = {
    "/content/drive/MyDrive/lora_adapters/sft_identity_v1": "outputs/lora_adapter",
    "/content/drive/MyDrive/lora_adapters/dpo_identity_v1": "outputs/dpo_adapter",
    "/content/drive/MyDrive/lora_adapters/reward_model_identity_v1": "outputs/reward_model",
}
for src, dst in restore.items():
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("已恢复:", dst)
    else:
        print("Drive 无备份, 跳过:", src)
print("\n环境就绪, 可以开始跑各阶段 notebook 了。")